In [ ]:
# @title Github Data Loader, Tokenizer Training, and Dataset Builder

import os

try:
    import google.colab
    REPO_URL = "https://github.com/wtheisen/nd-cse-10124-lectures.git"

    REPO_NAME = "/content/nd-cse-10124-lectures"
    L_PATH = "nd-cse-10124-lectures"

    %cd /content/
    !rm -r {REPO_NAME}

    # Clone repo
    if not os.path.exists(REPO_NAME):
        !git clone {REPO_URL}

        # cd into the data folder
        %cd {L_PATH}
        !pwd

        !pip install torch transformers regex tqdm

        !python scripts/import_gpt2_small.py \
  --out Datasets/gpt2_small_converted.pt \
  --tokenizer_dir Datasets/gpt2_tokenizer_assets

except ImportError:
    print("Unable to download repo, either:")
    print("\tA.) You're not on colab")
    print("\tB.) It has already been cloned")

!pwd

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

import irishGPT as iGPT
r_t = iGPT.tokenizer.Regex_Tokenizer()
r_t.load('Datasets/openweb10k_vocab.json')

dataset = iGPT.dataset.IrishChatDataset('Datasets/openweb10k.txt', r_t)

/content
rm: cannot remove '/content/nd-cse-10124-lectures': No such file or directory
Cloning into 'nd-cse-10124-lectures'...
remote: Enumerating objects: 396, done.
remote: Counting objects: 100% (129/129), done.
remote: Compressing objects: 100% (84/84), done.
remote: Total 396 (delta 75), reused 96 (delta 45), pack-reused 267 (from 1)
Receiving objects: 100% (396/396), 34.32 MiB | 27.98 MiB/s, done.
Resolving deltas: 100% (251/251), done.
/content/nd-cse-10124-lectures
/content/nd-cse-10124-lectures
/content/nd-cse-10124-lectures
device: cuda


In [ ]:
import torch
from irishGPT.irishChat import IrishChat
from irishGPT.gpt2_tokenizer import GPT2Tokenizer

# Paths
ckpt_path = "Datasets/gpt2_small_converted.pt"
vocab_path = "Datasets/gpt2_tokenizer_assets/vocab.json"
merges_path = "Datasets/gpt2_tokenizer_assets/merges.txt"

# Load tokenizer + model
tok = GPT2Tokenizer(vocab_path, merges_path)
model = IrishChat.gpt2_small()
model.load_converted_gpt2_checkpoint(ckpt_path)

# Prompt -> tokens
prompt = "Explain what attention does in one paragraph."
prompt_ids = tok.encode(prompt)

# Generate
out_ids = model.chat(
    prompt_ids,
    max_new_tokens=120,
    temperature=0.8,
    eos_token_id=50256,  # GPT-2 endoftext
)

# Decode
print(tok.decode(out_ids))

In [ ]:
# Mini ChatGPT-style UI for Colab (ipywidgets) — FIXED
# Assumes you already have:
#   - dataset.tokenizer with encode/decode
#   - chat.chat(prompt_tokens, max_new_tokens=..., temperature=...)

!pip -q install ipywidgets

from google.colab import output
output.enable_custom_widget_manager()

import ipywidgets as widgets
import html

# --------- STATE ----------
history = []  # list of (role, text), role in {"user","assistant"}

# --------- WIDGET STYLES (must be widgets.HTML, not IPython.display.HTML) ----------
style = widgets.HTML(value="""
<style>
.chat-wrap { font-family: system-ui, -apple-system, Segoe UI, Roboto, sans-serif; }
.chat-log  { height: 420px; overflow-y: auto; border: 1px solid #ddd; border-radius: 14px; padding: 12px; background: #fafafa; }
.msg { display: flex; margin: 10px 0; }
.bubble { max-width: 85%; padding: 10px 12px; border-radius: 14px; line-height: 1.35; white-space: pre-wrap; }
.user { justify-content: flex-end; }
.user .bubble { background: #dbeafe; border: 1px solid #bfdbfe; }
.assistant { justify-content: flex-start; }
.assistant .bubble { background: #ffffff; border: 1px solid #e5e7eb; }
.meta { font-size: 12px; color: #6b7280; margin-top: 6px; }
.row { display: flex; gap: 8px; margin-top: 10px; align-items: center; }
</style>
""")

# --------- UI ELEMENTS ----------
log = widgets.HTML(value="")
prompt = widgets.Text(
    placeholder="Message IrishGPT…",
    layout=widgets.Layout(width="60%")
)
send = widgets.Button(description="Send", button_style="primary")
clear_btn = widgets.Button(description="Clear")

temp = widgets.FloatSlider(
    value=0.8, min=0.1, max=1.5, step=0.05,
    description="Temp", continuous_update=False,
    layout=widgets.Layout(width="300px")
)

max_new = widgets.IntSlider(
    value=200, min=16, max=512, step=16,
    description="MaxNew", continuous_update=False,
    layout=widgets.Layout(width="320px")
)

status = widgets.HTML(value="<div class='meta'>Ready.</div>")

# --------- RENDERING ----------
def render_history():
    parts = ["<div class='chat-wrap'><div class='chat-log'>"]
    for role, text in history:
        safe = html.escape(text)
        cls = "user" if role == "user" else "assistant"
        parts.append(f"<div class='msg {cls}'><div class='bubble'>{safe}</div></div>")
    parts.append("</div></div>")
    log.value = "".join(parts)

def add_message(role, text):
    history.append((role, text))
    render_history()

# --------- GENERATION ----------
def generate_reply(user_text: str) -> str:
    prompt_tokens = dataset.tokenizer.encode("<|sos|>" + user_text + "<|eos|>")[:-1]
    out_tokens = chat.chat(
        prompt_tokens,
        max_new_tokens=int(max_new.value),
        temperature=float(temp.value),
    )
    return dataset.tokenizer.decode(out_tokens)

# --------- HANDLERS ----------
def on_send(_=None):
    user_text = prompt.value.strip()
    if not user_text:
        return
    prompt.value = ""

    add_message("user", user_text)
    status.value = "<div class='meta'>Generating…</div>"

    try:
        reply = generate_reply(user_text)
        add_message("assistant", reply)
        status.value = "<div class='meta'>Ready.</div>"
    except Exception as e:
        add_message("assistant", f"[error] {type(e).__name__}: {e}")
        status.value = "<div class='meta'>Error.</div>"

def on_clear(_=None):
    history.clear()
    render_history()
    status.value = "<div class='meta'>Cleared.</div>"

send.on_click(on_send)
clear_btn.on_click(on_clear)
prompt.on_submit(on_send)

# --------- LAYOUT ----------
controls = widgets.HBox(
    [send, clear_btn, temp, max_new],
    layout=widgets.Layout(width="100%")
)

ui = widgets.VBox(
    [style, log, prompt, controls, status],
    layout=widgets.Layout(
        width="40%",      # ← chat width
        margin="0 auto"   # ← center horizontally
    )
)


render_history()
display(ui)